# Vegetable price forecasting — conditional text rerun

## Leakage-safe BERTopic–FinBERT features and exploratory paired forecasting

This notebook implements the only defensible text analysis supported by the available
Google News file. It does **not** restore the manuscript's original general daily
forecasting claim.

Pre-specified design:

- A date is eligible only when all eight commodity outcomes are observed and the
  conservative text cutoff day contains at least one deduplicated full-text article.
- The 90 eligible dates are assigned chronologically: first 30 training, next 15
  validation, and final 45 test dates.
- BERTopic is fitted only on articles available through the last training cutoff day,
  then frozen for validation and test articles.
- Unavailable collection days are never encoded as zero-news days.
- The primary paired comparison is a pooled price-only Ridge model versus the same
  Ridge specification augmented with topic-conditioned news features.
- Model selection uses validation RMSE only. The 45 test dates remain untouched until
  final evaluation.
- The DM–HLN calculation is retained only as sensitivity evidence because the test
  dates are irregular and contain a long calendar gap.
- Volatility results are descriptive, with thresholds estimated from pre-validation
  price history.

Use a Colab GPU runtime. The notebook writes resumable translation, BERTopic, and
FinBERT checkpoints and packages all outputs into one ZIP file.


## 1. Runtime setup

In Google Colab, choose **Runtime → Change runtime type → GPU** before running.
PyTorch is left at Colab's installed version to avoid CUDA incompatibility.


In [ ]:
%pip install -q \
  "bertopic==0.16.4" \
  "sentence-transformers==3.3.1" \
  "transformers==4.47.1" \
  "accelerate==1.2.1" \
  "umap-learn==0.5.7" \
  "hdbscan==0.8.40" \
  "langid==1.1.6" \
  "safetensors==0.4.5" \
  "scikit-learn==1.5.2" \
  "scipy==1.14.1" \
  "pandas==2.2.3" \
  "seaborn==0.13.2"


In [ ]:
import gc
import hashlib
import json
import random
import subprocess
import sys
import zipfile
from pathlib import Path

import hdbscan
import langid
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from bertopic import BERTopic
from scipy import stats
from sentence_transformers import SentenceTransformer
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoModelForSequenceClassification, AutoTokenizer
from umap import UMAP
from sklearn.feature_extraction.text import CountVectorizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('Device:', DEVICE)
if DEVICE.type != 'cuda':
    raise RuntimeError('A GPU runtime is required for translation, BERTopic, and FinBERT.')


## 2. Load the corrected preprocessing package

Upload `Vegetable_Forecasting_Corrected_Preprocessing.zip` when prompted. The raw
source files are not modified.


In [ ]:
WORK_ROOT = Path('/content/vegetable_conditional_rerun')
INPUT_DIR = WORK_ROOT / 'input'
OUTPUT_DIR = WORK_ROOT / 'outputs'
MODEL_DIR = OUTPUT_DIR / 'bertopic_model'
for directory in [INPUT_DIR, OUTPUT_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import files
    colab_zip_path = Path('/content/Vegetable_Forecasting_Corrected_Preprocessing.zip')
    if colab_zip_path.exists():
        zip_path = colab_zip_path
    else:
        uploaded = files.upload()
        zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
        if len(zip_names) != 1:
            raise ValueError('Upload exactly one corrected preprocessing ZIP file.')
        zip_path = Path('/content') / zip_names[0]
except ImportError:
    zip_path = Path('Vegetable_Forecasting_Corrected_Preprocessing.zip')
    if not zip_path.exists():
        raise FileNotFoundError('Place the corrected preprocessing ZIP beside this notebook.')

with zipfile.ZipFile(zip_path) as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise RuntimeError(f'Corrupt ZIP member: {bad_member}')
    archive.extractall(INPUT_DIR)

required_files = {
    'manifest': 'preprocessing_manifest.json',
    'primary_news': 'news_primary_full_text_pretranslation.csv',
    'news_availability': 'news_daily_availability.csv',
    'price_daily': 'price_daily_causal.csv',
    'price_features': 'price_features_causal.csv',
}
resolved = {}
for key, filename in required_files.items():
    matches = list(INPUT_DIR.rglob(filename))
    if len(matches) != 1:
        raise FileNotFoundError(f'Expected exactly one {filename}; found {len(matches)}.')
    resolved[key] = matches[0]

manifest = json.loads(resolved['manifest'].read_text(encoding='utf-8'))
print('Input manifest status:', manifest.get('status'))
print('Primary full-text articles:', manifest['news']['primary_full_text_articles'])
print('Price items:', len(manifest['price']['items']))


## 3. Freeze the conditional 30/15/45 calendar

Eligibility is derived from the corrected package, not selected after seeing model
performance. `text_cutoff_day` is one day before the forecast origin and two calendar
days before the target.


In [ ]:
news = pd.read_csv(resolved['primary_news'])
news_availability = pd.read_csv(resolved['news_availability'])
price_daily = pd.read_csv(resolved['price_daily'])
price_features = pd.read_csv(resolved['price_features'])

news['published_day'] = pd.to_datetime(news['published_day']).dt.normalize()
news_availability['date'] = pd.to_datetime(news_availability['date']).dt.normalize()
price_daily['date'] = pd.to_datetime(price_daily['date']).dt.normalize()
price_features['target_date'] = pd.to_datetime(price_features['target_date']).dt.normalize()
price_features['forecast_origin'] = pd.to_datetime(price_features['forecast_origin']).dt.normalize()

if len(news) != manifest['news']['primary_full_text_articles']:
    raise ValueError('Primary news row count does not match the manifest.')
if news['article_identity'].duplicated().any():
    raise ValueError('Duplicate article identities remain in the primary corpus.')
if not news['document_type'].eq('full_text').all():
    raise ValueError('The primary corpus contains a non-full-text record.')

item_count = price_features['item'].nunique()
if item_count != 8:
    raise ValueError(f'Expected eight commodities, found {item_count}.')

common_targets = (
    price_features.loc[price_features['target_observed']]
    .groupby('target_date', as_index=False)
    .agg(n_items=('item', 'nunique'))
    .query('n_items == 8')
)
common_targets['forecast_origin'] = common_targets['target_date'] - pd.Timedelta(days=1)
common_targets['text_cutoff_day'] = common_targets['forecast_origin'] - pd.Timedelta(days=1)
availability = news_availability.rename(columns={'date': 'text_cutoff_day'})
calendar = common_targets.merge(availability, on='text_cutoff_day', how='left', validate='one_to_one')
count_columns = ['raw_rows', 'primary_full_text_articles', 'sensitivity_articles', 'title_only_articles']
calendar[count_columns] = calendar[count_columns].fillna(0).astype(int)
calendar['verified_primary_text'] = calendar['primary_full_text_articles'].gt(0)

conditional_calendar = calendar.loc[calendar['verified_primary_text']].sort_values('target_date').reset_index(drop=True)
if len(conditional_calendar) != 90:
    raise ValueError(f'Expected exactly 90 eligible targets; found {len(conditional_calendar)}.')

conditional_calendar['split'] = np.select(
    [conditional_calendar.index < 30, conditional_calendar.index < 45],
    ['train', 'validation'],
    default='test',
)
expected_sizes = {'train': 30, 'validation': 15, 'test': 45}
if conditional_calendar.groupby('split').size().to_dict() != expected_sizes:
    raise ValueError('Conditional calendar split sizes are incorrect.')

FIT_END_DATE = conditional_calendar.loc[
    conditional_calendar['split'].eq('train'), 'text_cutoff_day'
].max()
VALIDATION_START = conditional_calendar.loc[
    conditional_calendar['split'].eq('validation'), 'target_date'
].min()
TEST_START = conditional_calendar.loc[
    conditional_calendar['split'].eq('test'), 'target_date'
].min()

conditional_calendar.to_csv(OUTPUT_DIR / 'conditional_evaluation_calendar.csv', index=False)
split_summary = conditional_calendar.groupby('split', sort=False).agg(
    dates=('target_date', 'size'),
    first_target=('target_date', 'min'),
    last_target=('target_date', 'max'),
    primary_articles=('primary_full_text_articles', 'sum'),
).reset_index()
split_summary.to_csv(OUTPUT_DIR / 'conditional_split_summary.csv', index=False)

print('BERTopic fit cutoff:', FIT_END_DATE.date())
display(split_summary)
print('Maximum test calendar spacing (days):', int(
    conditional_calendar.loc[conditional_calendar['split'].eq('test'), 'target_date'].diff().dt.days.max()
))


## 4. Language detection and deterministic translation

`langid` is frozen for detection. English text is retained. Supported non-English text
is translated with `facebook/nllb-200-distilled-600M`. Failures are logged; the run
stops if more than 2% of documents are unusable.


In [ ]:
NLLB_MODEL_NAME = 'facebook/nllb-200-distilled-600M'
EMBEDDING_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
FINBERT_MODEL_NAME = 'ProsusAI/finbert'
TARGET_REGULAR_TOPICS = 19
MIN_TOPIC_SIZE = 20
TRANSLATION_BATCH_SIZE = 8
FINBERT_BATCH_SIZE = 16
MAX_SOURCE_TOKENS = 512
MAX_TRANSLATION_FAILURE_RATE = 0.02

LANGUAGE_CACHE = OUTPUT_DIR / 'language_detection.csv'
if LANGUAGE_CACHE.exists():
    detected = pd.read_csv(LANGUAGE_CACHE)
    news = news.merge(detected, on='article_identity', how='left', validate='one_to_one')
else:
    tqdm.pandas(desc='Detecting language')
    news['detected_language'] = news['model_text'].fillna('').astype(str).progress_map(
        lambda text: langid.classify(text[:4000])[0]
    )
    news[['article_identity', 'detected_language']].to_csv(LANGUAGE_CACHE, index=False)

language_counts = news['detected_language'].value_counts(dropna=False).rename_axis('language').reset_index(name='articles')
language_counts['share'] = language_counts['articles'] / len(news)
language_counts.to_csv(OUTPUT_DIR / 'language_distribution.csv', index=False)
display(language_counts.head(20))


In [ ]:
NLLB_LANGUAGE_CODES = {
    'ms': 'zsm_Latn', 'id': 'ind_Latn', 'vi': 'vie_Latn', 'zh': 'zho_Hans',
    'ta': 'tam_Taml', 'ar': 'arb_Arab', 'th': 'tha_Thai', 'hi': 'hin_Deva',
    'bn': 'ben_Beng', 'tl': 'tgl_Latn', 'jv': 'jav_Latn', 'ko': 'kor_Hang',
    'ja': 'jpn_Jpan', 'fr': 'fra_Latn', 'de': 'deu_Latn', 'es': 'spa_Latn',
    'pt': 'por_Latn', 'nl': 'nld_Latn', 'it': 'ita_Latn', 'ru': 'rus_Cyrl',
    'tr': 'tur_Latn', 'ur': 'urd_Arab', 'fa': 'pes_Arab', 'my': 'mya_Mymr',
}
TRANSLATION_CACHE = OUTPUT_DIR / 'primary_translation_checkpoint.csv'
TRANSLATION_FAILURE_LOG = OUTPUT_DIR / 'translation_failures.csv'

if TRANSLATION_CACHE.exists():
    cached = pd.read_csv(TRANSLATION_CACHE)
    needed = {'article_identity', 'translated_text', 'translation_status', 'translation_truncated'}
    if not needed.issubset(cached.columns):
        raise ValueError('Translation checkpoint is missing required columns.')
    news = news.merge(cached[list(needed)], on='article_identity', how='left', validate='one_to_one')
else:
    news['translated_text'] = pd.NA
    news['translation_status'] = 'pending'
    news['translation_truncated'] = False
    english = news['detected_language'].eq('en')
    news.loc[english, 'translated_text'] = news.loc[english, 'model_text']
    news.loc[english, 'translation_status'] = 'english_original'

pending_languages = sorted(news.loc[news['translation_status'].eq('pending'), 'detected_language'].dropna().unique())
unsupported = [language for language in pending_languages if language not in NLLB_LANGUAGE_CODES]
if unsupported:
    mask = news['detected_language'].isin(unsupported) & news['translation_status'].eq('pending')
    news.loc[mask, 'translation_status'] = 'unsupported_language'

languages_to_translate = [language for language in pending_languages if language in NLLB_LANGUAGE_CODES]
if languages_to_translate:
    tokenizer = AutoTokenizer.from_pretrained(NLLB_MODEL_NAME, src_lang='eng_Latn')
    dtype = torch.float16 if DEVICE.type == 'cuda' else torch.float32
    translator = AutoModelForSeq2SeqLM.from_pretrained(NLLB_MODEL_NAME, torch_dtype=dtype).to(DEVICE).eval()
    english_bos_id = tokenizer.convert_tokens_to_ids('eng_Latn')

    for language in languages_to_translate:
        tokenizer.src_lang = NLLB_LANGUAGE_CODES[language]
        indices = news.index[
            news['detected_language'].eq(language) & news['translation_status'].eq('pending')
        ].tolist()
        print(f'Translating {len(indices)} documents detected as {language}')
        for start in tqdm(range(0, len(indices), TRANSLATION_BATCH_SIZE), desc=language):
            batch_indices = indices[start:start + TRANSLATION_BATCH_SIZE]
            texts = news.loc[batch_indices, 'model_text'].fillna('').astype(str).tolist()
            try:
                token_lists = tokenizer(texts, add_special_tokens=True, truncation=False)['input_ids']
                truncated = [len(tokens) > MAX_SOURCE_TOKENS for tokens in token_lists]
                encoded = tokenizer(
                    texts, return_tensors='pt', padding=True, truncation=True,
                    max_length=MAX_SOURCE_TOKENS
                ).to(DEVICE)
                with torch.inference_mode():
                    generated = translator.generate(
                        **encoded, forced_bos_token_id=english_bos_id,
                        max_new_tokens=512, num_beams=1
                    )
                translated = tokenizer.batch_decode(generated, skip_special_tokens=True)
                news.loc[batch_indices, 'translated_text'] = translated
                news.loc[batch_indices, 'translation_status'] = 'translated_nllb'
                news.loc[batch_indices, 'translation_truncated'] = truncated
            except Exception as exc:
                news.loc[batch_indices, 'translation_status'] = f'failed:{type(exc).__name__}'
            if (start // TRANSLATION_BATCH_SIZE + 1) % 25 == 0:
                news[['article_identity', 'translated_text', 'translation_status', 'translation_truncated']].to_csv(
                    TRANSLATION_CACHE, index=False
                )
    del translator, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

news[['article_identity', 'translated_text', 'translation_status', 'translation_truncated']].to_csv(
    TRANSLATION_CACHE, index=False
)
usable_translation = news['translation_status'].isin(['english_original', 'translated_nllb'])
translation_failure_rate = float(1 - usable_translation.mean())
news.loc[~usable_translation, [
    'article_identity', 'published_day', 'detected_language', 'translation_status', 'title_clean'
]].to_csv(TRANSLATION_FAILURE_LOG, index=False)

print(news['translation_status'].value_counts(dropna=False))
print('Translation unusable rate:', f'{translation_failure_rate:.2%}')
if translation_failure_rate > MAX_TRANSLATION_FAILURE_RATE:
    raise RuntimeError('Translation failure rate exceeds 2%; review translation_failures.csv.')

model_news = news.loc[usable_translation].copy()
model_news['translated_text'] = model_news['translated_text'].fillna('').astype(str)
if model_news['translated_text'].str.strip().eq('').any():
    raise ValueError('Usable translation records contain empty text.')

post_translation_days = set(model_news['published_day'].unique())
missing_cutoffs = conditional_calendar.loc[
    ~conditional_calendar['text_cutoff_day'].isin(post_translation_days), 'text_cutoff_day'
].drop_duplicates().sort_values()
if len(missing_cutoffs):
    raise RuntimeError(
        'Translation removed every usable article from one or more pre-specified cutoff days: '
        + ', '.join(missing_cutoffs.dt.strftime('%Y-%m-%d'))
    )


## 5. Fit BERTopic on training-period news only

The fit boundary is the latest text cutoff used by the 30 training targets. Articles
after that boundary are assigned using `transform()` only.


In [ ]:
fit_mask = model_news['published_day'].le(FIT_END_DATE)
if not fit_mask.any() or fit_mask.all():
    raise ValueError('Invalid BERTopic fit/transform partition.')

fit_documents = model_news.loc[fit_mask, 'translated_text'].tolist()
transform_documents = model_news.loc[~fit_mask, 'translated_text'].tolist()
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=str(DEVICE))
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=SEED)
hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=MIN_TOPIC_SIZE, metric='euclidean',
    cluster_selection_method='leaf', prediction_data=True
)
vectorizer_model = CountVectorizer(stop_words='english', ngram_range=(1, 2), min_df=1)
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    nr_topics=TARGET_REGULAR_TOPICS + 1,
    calculate_probabilities=False,
    verbose=True,
)

fit_topics, _ = topic_model.fit_transform(fit_documents)
transformed_topics, _ = topic_model.transform(transform_documents)
model_news['topic'] = -999
model_news.loc[fit_mask, 'topic'] = fit_topics
model_news.loc[~fit_mask, 'topic'] = transformed_topics
model_news['topic'] = model_news['topic'].astype(int)

topic_info = topic_model.get_topic_info().copy()
topic_info['manual_label'] = ''
topic_info.to_csv(OUTPUT_DIR / 'bertopic_topic_summary.csv', index=False)

regular_topics = sorted(topic for topic in model_news['topic'].unique() if topic != -1)
outlier_rate = float(model_news['topic'].eq(-1).mean())
review_rows = []
for topic in regular_topics:
    representatives = topic_model.get_representative_docs(topic) or []
    if representatives and not isinstance(representatives[0], str):
        representatives = model_news.loc[
            fit_mask & model_news['topic'].eq(topic), 'translated_text'
        ].head(3).tolist()
    for rank, representative_text in enumerate(representatives, start=1):
        match = model_news.loc[model_news['translated_text'].eq(representative_text)].head(1)
        review_rows.append({
            'topic': int(topic),
            'representative_rank': rank,
            'published_day': match['published_day'].iloc[0] if len(match) else pd.NaT,
            'title_clean': match['title_clean'].iloc[0] if len(match) else '',
            'representative_text': representative_text,
            'proposed_manual_label': '',
        })
topic_review = pd.DataFrame(review_rows)
topic_review.to_csv(OUTPUT_DIR / 'topic_review_samples.csv', index=False)
topic_model.save(
    str(MODEL_DIR), serialization='safetensors', save_ctfidf=True,
    save_embedding_model=EMBEDDING_MODEL_NAME
)

print('Fit documents:', len(fit_documents))
print('Transformed documents:', len(transform_documents))
print('Regular topics:', len(regular_topics), regular_topics)
print('Topic -1 rate:', f'{outlier_rate:.2%}')
display(topic_info.head(25))
display(topic_review.head(30))


### Mandatory topic-review gate

Inspect `bertopic_topic_summary.csv` and `topic_review_samples.csv`. Confirm that the
topics are interpretable enough for an exploratory analysis and that topic `-1` is not
used as a predictive topic. Then set `TOPIC_REVIEW_APPROVED = True` and continue.


In [ ]:
TOPIC_REVIEW_APPROVED = False

if not TOPIC_REVIEW_APPROVED:
    raise RuntimeError(
        'TOPIC REVIEW REQUIRED: inspect the displayed topic summary and representative '
        'articles, then set TOPIC_REVIEW_APPROVED=True. Do not select topics using '
        'validation or test forecasting performance.'
    )


## 6. FinBERT article sentiment

FinBERT is fixed and not fine-tuned. Article sentiment is
`P(positive) - P(negative)`. Domain mismatch between financial news and agricultural
news remains a limitation and must be acknowledged in the manuscript.


In [ ]:
FINBERT_CACHE = OUTPUT_DIR / 'finbert_checkpoint.csv'
FINBERT_FAILURE_LOG = OUTPUT_DIR / 'finbert_failures.csv'
finbert_tokenizer = AutoTokenizer.from_pretrained(FINBERT_MODEL_NAME)
finbert_model = AutoModelForSequenceClassification.from_pretrained(FINBERT_MODEL_NAME).to(DEVICE).eval()
id2label = {int(k): str(v).lower() for k, v in finbert_model.config.id2label.items()}
label2index = {label: index for index, label in id2label.items()}
if not {'positive', 'negative', 'neutral'}.issubset(label2index):
    raise ValueError(f'Unexpected FinBERT labels: {id2label}')

if FINBERT_CACHE.exists():
    finbert_cache = pd.read_csv(FINBERT_CACHE)
else:
    finbert_cache = pd.DataFrame(columns=[
        'article_identity', 'positive_probability', 'negative_probability',
        'neutral_probability', 'finbert_status', 'finbert_truncated'
    ])

completed = set(finbert_cache.loc[finbert_cache['finbert_status'].eq('ok'), 'article_identity'])
pending = model_news.loc[~model_news['article_identity'].isin(completed)].copy()
new_results = []
for start in tqdm(range(0, len(pending), FINBERT_BATCH_SIZE), desc='FinBERT'):
    batch = pending.iloc[start:start + FINBERT_BATCH_SIZE]
    texts = batch['translated_text'].tolist()
    try:
        token_lists = finbert_tokenizer(texts, add_special_tokens=True, truncation=False)['input_ids']
        truncated = [len(tokens) > 512 for tokens in token_lists]
        encoded = finbert_tokenizer(
            texts, return_tensors='pt', padding=True, truncation=True, max_length=512
        ).to(DEVICE)
        with torch.inference_mode():
            probabilities = torch.softmax(finbert_model(**encoded).logits, dim=-1).cpu().numpy()
        for row_index, (_, row) in enumerate(batch.iterrows()):
            new_results.append({
                'article_identity': row['article_identity'],
                'positive_probability': float(probabilities[row_index, label2index['positive']]),
                'negative_probability': float(probabilities[row_index, label2index['negative']]),
                'neutral_probability': float(probabilities[row_index, label2index['neutral']]),
                'finbert_status': 'ok',
                'finbert_truncated': bool(truncated[row_index]),
            })
    except Exception as exc:
        for article_identity in batch['article_identity']:
            new_results.append({
                'article_identity': article_identity,
                'positive_probability': np.nan,
                'negative_probability': np.nan,
                'neutral_probability': np.nan,
                'finbert_status': f'failed:{type(exc).__name__}',
                'finbert_truncated': False,
            })
    if (start // FINBERT_BATCH_SIZE + 1) % 25 == 0:
        checkpoint = pd.concat([finbert_cache, pd.DataFrame(new_results)], ignore_index=True)
        checkpoint.drop_duplicates('article_identity', keep='last').to_csv(FINBERT_CACHE, index=False)

finbert_cache = pd.concat([finbert_cache, pd.DataFrame(new_results)], ignore_index=True)
finbert_cache = finbert_cache.drop_duplicates('article_identity', keep='last')
finbert_cache.to_csv(FINBERT_CACHE, index=False)
model_news = model_news.merge(finbert_cache, on='article_identity', how='left', validate='one_to_one')
failed_finbert = ~model_news['finbert_status'].eq('ok')
model_news.loc[failed_finbert, [
    'article_identity', 'published_day', 'finbert_status', 'title_clean'
]].to_csv(FINBERT_FAILURE_LOG, index=False)
if failed_finbert.any():
    raise RuntimeError(f'FinBERT failed for {int(failed_finbert.sum())} articles.')

probability_sum = model_news[['positive_probability', 'negative_probability', 'neutral_probability']].sum(axis=1)
if not np.allclose(probability_sum, 1.0, atol=1e-4):
    raise ValueError('FinBERT probabilities do not sum to one.')
model_news['sentiment_score'] = model_news['positive_probability'] - model_news['negative_probability']
print(model_news['sentiment_score'].describe())


## 7. Build exact-day topic features without filling unavailable days

Only same-cutoff-day article counts and sentiment are used. Rolling and exponentially
decayed text features are deliberately omitted because collection gaps cannot be
distinguished from genuine zero-news days.


In [ ]:
article_columns = [
    'article_identity', 'published_day', 'publisher', 'source_query_topics',
    'detected_language', 'translation_status', 'translation_truncated',
    'topic', 'positive_probability', 'negative_probability',
    'neutral_probability', 'sentiment_score', 'finbert_truncated'
]
model_news[article_columns].sort_values(['published_day', 'article_identity']).to_csv(
    OUTPUT_DIR / 'article_topics_sentiment.csv', index=False
)

observed_days = pd.Index(sorted(model_news['published_day'].unique()), name='news_day')
daily_features = pd.DataFrame(index=observed_days)
all_group = model_news.groupby('published_day')
daily_features['all_article_count'] = all_group.size().reindex(observed_days).astype(int)
daily_features['all_sentiment_mean'] = all_group['sentiment_score'].mean().reindex(observed_days)
daily_features['positive_share'] = all_group['positive_probability'].apply(lambda s: float((s > 0.5).mean())).reindex(observed_days)
daily_features['negative_share'] = all_group['negative_probability'].apply(lambda s: float((s > 0.5).mean())).reindex(observed_days)
daily_features['topic_minus_one_count'] = (
    model_news.loc[model_news['topic'].eq(-1)].groupby('published_day').size().reindex(observed_days, fill_value=0).astype(int)
)

regular = model_news.loc[model_news['topic'].ne(-1)].copy()
daily_grouped = regular.groupby(['published_day', 'topic']).agg(
    sentiment_sum=('sentiment_score', 'sum'),
    article_count=('article_identity', 'size'),
).reset_index()
for topic in regular_topics:
    topic_daily = daily_grouped.loc[daily_grouped['topic'].eq(topic)].set_index('published_day')
    counts = topic_daily['article_count'].reindex(observed_days, fill_value=0).astype(int)
    sums = topic_daily['sentiment_sum'].reindex(observed_days, fill_value=0.0)
    means = np.divide(
        sums.to_numpy(), counts.to_numpy(),
        out=np.zeros(len(observed_days), dtype=float),
        where=counts.to_numpy() > 0,
    )
    prefix = f'topic_{int(topic):02d}'
    daily_features[f'{prefix}_article_count'] = counts.to_numpy()
    daily_features[f'{prefix}_sentiment_mean'] = means

daily_features = daily_features.reset_index()
daily_features.to_csv(OUTPUT_DIR / 'daily_topic_sentiment_features_conditional.csv', index=False)
if daily_features.drop(columns='news_day').isna().any().any():
    raise ValueError('Daily text features contain missing values on observed news days.')

aligned_dates = conditional_calendar.merge(
    daily_features, left_on='text_cutoff_day', right_on='news_day',
    how='left', validate='one_to_one'
)
if aligned_dates['news_day'].isna().any() or aligned_dates['all_article_count'].le(0).any():
    raise RuntimeError('A pre-specified conditional date lacks usable post-translation news.')
aligned_dates = aligned_dates.drop(columns=['news_day'])
aligned_dates.to_csv(OUTPUT_DIR / 'conditional_text_features_by_date.csv', index=False)

model_data = price_features.merge(
    aligned_dates, on=['target_date', 'forecast_origin'], how='inner', validate='many_to_one'
)
if len(model_data) != 90 * 8:
    raise ValueError(f'Expected 720 item-date rows, found {len(model_data)}.')
if model_data.groupby(['split', 'target_date'])['item'].nunique().ne(8).any():
    raise ValueError('Not every conditional date contains all eight commodities.')
model_data.to_csv(OUTPUT_DIR / 'conditional_model_ready_dataset.csv', index=False)
print('Model-ready rows:', len(model_data))
display(model_data.groupby('split').agg(rows=('item', 'size'), dates=('target_date', 'nunique')))


## 8. Pooled Ridge comparison

The low-complexity pooled model uses commodity indicators and the same causal price
features in both specifications. The hybrid adds exact-day topic counts and sentiment.
Separate Ridge penalties are chosen on the 15 validation dates, then each model is
refitted on training plus validation and evaluated once on the 45 test dates.


In [ ]:
excluded = {
    'target_date', 'forecast_origin', 'item', 'y_true', 'target_observed',
    'n_items', 'text_cutoff_day', 'raw_rows', 'primary_full_text_articles',
    'sensitivity_articles', 'title_only_articles', 'verified_primary_text', 'split',
}
price_columns = [
    column for column in price_features.columns
    if column not in {'target_date', 'forecast_origin', 'item', 'y_true', 'target_observed'}
]
text_columns = [
    column for column in daily_features.columns
    if column not in {'news_day', 'topic_minus_one_count'}
]
count_columns = [column for column in text_columns if 'count' in column]
for column in count_columns:
    model_data[f'log1p_{column}'] = np.log1p(model_data[column].astype(float))
text_model_columns = [f'log1p_{column}' if column in count_columns else column for column in text_columns]

def make_pipeline(numeric_columns, alpha):
    preprocessor = ColumnTransformer([
        ('numeric', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), numeric_columns),
        ('item', OneHotEncoder(handle_unknown='ignore', drop=None), ['item']),
    ])
    return Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', Ridge(alpha=alpha)),
    ])

alpha_grid = [0.1, 1.0, 10.0, 100.0, 1000.0]
train = model_data.loc[model_data['split'].eq('train')].copy()
validation = model_data.loc[model_data['split'].eq('validation')].copy()
test = model_data.loc[model_data['split'].eq('test')].copy()

specifications = {
    'price_only_ridge': price_columns,
    'text_enhanced_ridge': price_columns + text_model_columns,
}
selection_rows = []
selected_alpha = {}
for model_name, numeric_columns in specifications.items():
    for alpha in alpha_grid:
        candidate = make_pipeline(numeric_columns, alpha)
        candidate.fit(train[numeric_columns + ['item']], train['y_true'])
        prediction = candidate.predict(validation[numeric_columns + ['item']])
        rmse = mean_squared_error(validation['y_true'], prediction) ** 0.5
        selection_rows.append({'model': model_name, 'alpha': alpha, 'validation_rmse': rmse})
    candidates = pd.DataFrame(selection_rows).query('model == @model_name').sort_values(['validation_rmse', 'alpha'])
    selected_alpha[model_name] = float(candidates.iloc[0]['alpha'])

model_selection = pd.DataFrame(selection_rows)
model_selection['selected'] = model_selection.apply(
    lambda row: row['alpha'] == selected_alpha[row['model']], axis=1
)
model_selection.to_csv(OUTPUT_DIR / 'model_selection_validation.csv', index=False)
display(model_selection.sort_values(['model', 'validation_rmse']))

development = pd.concat([train, validation], ignore_index=True)
prediction_frames = []
for model_name, numeric_columns in specifications.items():
    fitted = make_pipeline(numeric_columns, selected_alpha[model_name])
    fitted.fit(development[numeric_columns + ['item']], development['y_true'])
    pred = test[['target_date', 'forecast_origin', 'text_cutoff_day', 'item', 'y_true']].copy()
    pred['model'] = model_name
    pred['y_pred'] = fitted.predict(test[numeric_columns + ['item']])
    prediction_frames.append(pred)

naive = test[['target_date', 'forecast_origin', 'text_cutoff_day', 'item', 'y_true']].copy()
naive['model'] = 'naive_lag_1'
naive['y_pred'] = test['price_lag_1'].to_numpy()
prediction_frames.append(naive)
predictions = pd.concat(prediction_frames, ignore_index=True)
predictions.to_csv(OUTPUT_DIR / 'conditional_test_predictions.csv', index=False)

pair_check = predictions.loc[predictions['model'].isin(['price_only_ridge', 'text_enhanced_ridge'])].pivot(
    index=['target_date', 'forecast_origin', 'item', 'y_true'], columns='model', values='y_pred'
).reset_index()
if len(pair_check) != 45 * 8:
    raise ValueError('The paired Ridge comparison is not aligned on 45 dates × 8 commodities.')
print('Paired test rows:', len(pair_check))


## 9. Metrics and paired sensitivity analysis

The DM–HLN results are not treated as confirmatory because the 45 test observations
are irregularly spaced. Holm-adjusted p-values, mean loss differences, and win rates
are reported to make the sensitivity analysis transparent.


In [ ]:
def smape(y_true, y_pred):
    denominator = np.abs(y_true) + np.abs(y_pred)
    values = np.divide(
        2 * np.abs(y_pred - y_true), denominator,
        out=np.zeros_like(np.asarray(y_true, dtype=float)),
        where=denominator != 0,
    )
    return float(np.mean(values) * 100)

metric_rows = []
for (model_name, item), group in predictions.groupby(['model', 'item']):
    metric_rows.append({
        'scope': 'commodity', 'item': item, 'model': model_name, 'n': len(group),
        'mae': mean_absolute_error(group['y_true'], group['y_pred']),
        'rmse': mean_squared_error(group['y_true'], group['y_pred']) ** 0.5,
        'smape_pct': smape(group['y_true'].to_numpy(), group['y_pred'].to_numpy()),
    })
for model_name, group in predictions.groupby('model'):
    metric_rows.append({
        'scope': 'pooled', 'item': 'ALL', 'model': model_name, 'n': len(group),
        'mae': mean_absolute_error(group['y_true'], group['y_pred']),
        'rmse': mean_squared_error(group['y_true'], group['y_pred']) ** 0.5,
        'smape_pct': smape(group['y_true'].to_numpy(), group['y_pred'].to_numpy()),
    })
metrics = pd.DataFrame(metric_rows)
metrics.to_csv(OUTPUT_DIR / 'conditional_test_metrics.csv', index=False)
display(metrics.loc[metrics['scope'].eq('pooled')].sort_values('rmse'))

def dm_hln(loss_baseline, loss_hybrid, h=1):
    differential = np.asarray(loss_baseline) - np.asarray(loss_hybrid)
    n = len(differential)
    if n < 3 or np.isclose(differential.std(ddof=1), 0):
        return np.nan, np.nan
    dm = differential.mean() / (differential.std(ddof=1) / np.sqrt(n))
    correction = np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    statistic = dm * correction
    p_value = 2 * stats.t.sf(abs(statistic), df=n - 1)
    return float(statistic), float(p_value)

def holm_adjust(p_values):
    p = np.asarray(p_values, dtype=float)
    order = np.argsort(p)
    adjusted_sorted = np.maximum.accumulate((len(p) - np.arange(len(p))) * p[order])
    adjusted = np.empty_like(adjusted_sorted)
    adjusted[order] = np.minimum(adjusted_sorted, 1.0)
    return adjusted

dm_rows = []
for item, group in pair_check.groupby('item'):
    baseline_loss = (group['y_true'] - group['price_only_ridge']) ** 2
    hybrid_loss = (group['y_true'] - group['text_enhanced_ridge']) ** 2
    statistic, p_value = dm_hln(baseline_loss, hybrid_loss, h=1)
    dm_rows.append({
        'item': item,
        'n_irregular_test_dates': len(group),
        'dm_hln_statistic': statistic,
        'p_value_unadjusted': p_value,
        'mean_squared_loss_difference_baseline_minus_hybrid': float((baseline_loss - hybrid_loss).mean()),
        'hybrid_win_rate': float((hybrid_loss < baseline_loss).mean()),
        'interpretation': 'Sensitivity only: irregular conditional dates',
    })
dm_results = pd.DataFrame(dm_rows)
valid = dm_results['p_value_unadjusted'].notna()
dm_results['p_value_holm'] = np.nan
dm_results.loc[valid, 'p_value_holm'] = holm_adjust(dm_results.loc[valid, 'p_value_unadjusted'])
dm_results.to_csv(OUTPUT_DIR / 'dm_hln_irregular_date_sensitivity.csv', index=False)
display(dm_results)


## 10. Training-derived volatility descriptions

For each commodity, high volatility is defined using the 75th percentile of absolute
one-day returns in observed price history before validation begins. The test subsets
are descriptive; cells with fewer than 10 observations are flagged as too small.


In [ ]:
history = price_features.loc[
    price_features['target_date'].lt(VALIDATION_START) & price_features['target_observed']
].copy()
history['abs_return'] = np.abs(history['y_true'] / history['price_lag_1'] - 1)
thresholds = history.groupby('item')['abs_return'].quantile(0.75).rename('high_volatility_threshold')

test_with_volatility = test[['target_date', 'item', 'y_true', 'price_lag_1']].copy()
test_with_volatility['abs_return'] = np.abs(test_with_volatility['y_true'] / test_with_volatility['price_lag_1'] - 1)
test_with_volatility = test_with_volatility.join(thresholds, on='item')
test_with_volatility['volatility_regime'] = np.where(
    test_with_volatility['abs_return'].ge(test_with_volatility['high_volatility_threshold']),
    'high', 'lower'
)

volatility_rows = []
for (item, regime), dates in test_with_volatility.groupby(['item', 'volatility_regime']):
    keys = dates[['target_date', 'item']]
    for model_name in ['price_only_ridge', 'text_enhanced_ridge']:
        model_predictions = predictions.loc[predictions['model'].eq(model_name)].merge(
            keys, on=['target_date', 'item'], how='inner', validate='one_to_one'
        )
        volatility_rows.append({
            'item': item, 'volatility_regime': regime, 'model': model_name,
            'n': len(model_predictions),
            'mae': mean_absolute_error(model_predictions['y_true'], model_predictions['y_pred']),
            'rmse': mean_squared_error(model_predictions['y_true'], model_predictions['y_pred']) ** 0.5,
            'inferential_status': 'descriptive_only' if len(model_predictions) >= 10 else 'descriptive_too_small_n_lt_10',
        })
volatility_results = pd.DataFrame(volatility_rows)
volatility_results.to_csv(OUTPUT_DIR / 'volatility_conditioned_descriptive_metrics.csv', index=False)
thresholds.reset_index().to_csv(OUTPUT_DIR / 'training_volatility_thresholds.csv', index=False)
display(volatility_results)


## 11. Diagnostics, safeguards, and output package


In [ ]:
plt.figure(figsize=(12, 4))
split_colors = {'train': '#4C78A8', 'validation': '#F58518', 'test': '#E45756'}
for split, group in conditional_calendar.groupby('split', sort=False):
    plt.scatter(group['target_date'], np.repeat(split, len(group)), s=30, color=split_colors[split], label=split)
plt.title('Conditional evaluation dates (unavailable days remain absent)')
plt.xlabel('Target date')
plt.ylabel('Split')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'conditional_evaluation_dates.png', dpi=200)
plt.show()

pooled_metrics = metrics.loc[metrics['scope'].eq('pooled')].copy()
plt.figure(figsize=(8, 4))
sns.barplot(data=pooled_metrics, x='model', y='rmse', color='#4C78A8')
plt.title('Exploratory conditional test RMSE')
plt.xlabel('Model')
plt.ylabel('RMSE')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'conditional_test_rmse.png', dpi=200)
plt.show()

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

diagnostics = {
    'status': 'exploratory_conditional_text_analysis',
    'seed': SEED,
    'fit_end_date': str(FIT_END_DATE.date()),
    'split_sizes_dates': {'train': 30, 'validation': 15, 'test': 45},
    'commodities': int(item_count),
    'training_rows_pooled': int(len(train)),
    'validation_rows_pooled': int(len(validation)),
    'test_rows_pooled': int(len(test)),
    'test_first_target': str(test['target_date'].min().date()),
    'test_last_target': str(test['target_date'].max().date()),
    'test_max_calendar_spacing_days': int(
        conditional_calendar.loc[conditional_calendar['split'].eq('test'), 'target_date'].diff().dt.days.max()
    ),
    'translation_failure_rate': translation_failure_rate,
    'fit_articles': int(fit_mask.sum()),
    'transformed_articles': int((~fit_mask).sum()),
    'regular_topics': [int(topic) for topic in regular_topics],
    'topic_minus_one_rate': outlier_rate,
    'selected_alpha': selected_alpha,
    'models': {
        'translation': NLLB_MODEL_NAME,
        'topic_embedding': EMBEDDING_MODEL_NAME,
        'sentiment': FINBERT_MODEL_NAME,
        'forecaster': 'pooled Ridge with commodity one-hot indicators',
    },
    'claims_guard': {
        'general_daily_forecasting_claim_supported': False,
        'conditional_on_verified_news_exploratory_claim_only': True,
        'dm_hln_confirmatory': False,
        'volatility_inferential': False,
    },
    'integrity_checks': {
        'all_evaluation_dates_have_verified_primary_text': bool(conditional_calendar['verified_primary_text'].all()),
        'topic_fit_precedes_validation': bool(FIT_END_DATE < conditional_calendar.loc[conditional_calendar['split'].eq('validation'), 'text_cutoff_day'].min()),
        'test_pairs_aligned': bool(len(pair_check) == 45 * 8),
        'no_unavailable_days_zero_filled': True,
        'topic_minus_one_excluded_from_topic_predictors': True,
    },
    'input_hashes': {key: sha256(path) for key, path in resolved.items()},
}
if not all(diagnostics['integrity_checks'].values()):
    raise RuntimeError(f'Integrity check failed: {diagnostics["integrity_checks"]}')

(OUTPUT_DIR / 'conditional_rerun_diagnostics.json').write_text(json.dumps(diagnostics, indent=2), encoding='utf-8')
subprocess.run(
    [sys.executable, '-m', 'pip', 'freeze'], check=True,
    stdout=(OUTPUT_DIR / 'requirements_freeze.txt').open('w', encoding='utf-8')
)
print(json.dumps(diagnostics, indent=2))

RESULTS_ZIP = WORK_ROOT / 'Vegetable_Forecasting_Conditional_Rerun_Outputs.zip'
with zipfile.ZipFile(RESULTS_ZIP, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=9) as archive:
    for path in sorted(OUTPUT_DIR.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(OUTPUT_DIR))
with zipfile.ZipFile(RESULTS_ZIP) as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise RuntimeError(f'Output ZIP verification failed at {bad_member}')
print('Created:', RESULTS_ZIP)
print('Size (MB):', round(RESULTS_ZIP.stat().st_size / 1024 / 1024, 2))

try:
    from google.colab import files
    files.download(str(RESULTS_ZIP))
except ImportError:
    pass


## Interpretation gate

The outputs may support only a statement of the form:

> On the subset of dates with retained full-text news dated on the required cutoff day, the text-enhanced
> pooled Ridge model showed [direction and magnitude] relative to an otherwise matched
> price-only Ridge model.

They do not support a claim that text features improve forecasting on arbitrary daily
dates. Report the 30/15/45 split, the irregular test calendar, the longest gap, topic
outlier rate, selected Ridge penalties, all eight commodity-level effects, adjusted
p-values, and realised-movement subset sizes. Do not select or relabel topics based on test
performance.
